In [1]:
import pandas as pd

ativos = pd.read_csv("ativos.csv", sep=";")

vulnerabilidades = pd.read_csv("vulnerabilidades.csv", sep=";")

print("Ativos:", ativos.shape)

print("Vulnerabilidades:", vulnerabilidades.shape)



Ativos: (200, 7)
Vulnerabilidades: (2000, 8)


In [2]:
ativos.head()

,ativo_id,sistema,criticidade,dominio,ambiente,localidade,status_ativo
0,A0001,Sistema_1,Crítica,Dominio_2,Homologação,São Paulo,Em desativação
1,A0002,Sistema_2,Média,Dominio_3,Produção,São Paulo,Em desativação
2,A0003,Sistema_3,Média,Dominio_4,Homologação,São Paulo,Ativo
3,A0004,Sistema_4,Alta,Dominio_5,Homologação,Campinas,Ativo
4,A0005,Sistema_5,Alta,Dominio_6,Produção,Recife,Ativo


In [3]:
vulnerabilidades.head()

/usr/local/lib/python3.13/dist-packages/google/colab/_dataframe_summarizer.py:88: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  cast_date_col = pd.to_datetime(column, errors="coerce")


,vuln_id,ativo_id,severidade,origem,data_abertura,data_correcao,status,cvss_score
0,V00001,A0078,Baixa,Cloud Security,21/03/2026,NaN,Em tratamento,2.8
1,V00002,A0060,Alta,Qualys,01/06/2026,NaN,Em tratamento,7.5
2,V00003,A0080,Alta,Veracode,18/08/2026,NaN,Aberta,1.7
3,V00004,A0086,Alta,Qualys,16/06/2026,NaN,Aberta,2.5
4,V00005,A0089,Alta,Cloud Security,12/02/2026,NaN,Em tratamento,4.4


In [4]:
ativos["criticidade"].value_counts(dropna=False)

,count
criticidade,
Baixa,56
Média,46
Crítica,45
Alta,42
media,4
,3
BaixA,2
critica,1
NaN,1


In [5]:
ativos['criticidade_tratada'] = (

    ativos['criticidade']

    .astype(str)

    .str.strip()

    .str.lower()

    .replace({

        'critica': 'Crítica',

        'crítica': 'Crítica',

        'alta': 'Alta',

        'media': 'Média',

        'média': 'Média',

        'baixa': 'Baixa',

        'n/a': 'Não Informado',

        '': 'Não Informado'

    })

)

ativos['criticidade_tratada'].value_counts(dropna=False)



,count
criticidade_tratada,
Baixa,58
Média,50
Crítica,46
Alta,42
Não Informado,3
nan,1


In [6]:
ativos['criticidade_tratada'] = ativos['criticidade_tratada'].replace({'nan': 'Não Informado'})

ativos['criticidade_tratada'].value_counts(dropna=False)

,count
criticidade_tratada,
Baixa,58
Média,50
Crítica,46
Alta,42
Não Informado,4


In [8]:
vulnerabilidades['severidade'].value_counts(dropna=False)

,count
severidade,
Média,489
Crítica,487
Alta,472
Baixa,463
NaN,26
media,16
BaixA,16
ALTA,15
critica,13


In [9]:
vulnerabilidades['severidade_tratada'] = (

    vulnerabilidades['severidade']

    .astype(str)

    .str.strip()

    .str.lower()

    .replace({

        'critica': 'Crítica',

        'crítica': 'Crítica',

        'alta': 'Alta',

        'alto': 'Alta',

        'media': 'Média',

        'média': 'Média',

        'baixa': 'Baixa',

        'n/a': 'Não Informado',

        'nan': 'Não Informado',

        '': 'Não Informado'

    })

)

vulnerabilidades['severidade_tratada'].value_counts(dropna=False)

,count
severidade_tratada,
Média,506
Crítica,501
Alta,488
Baixa,479
Não Informado,26


In [10]:
vulnerabilidades['origem_tratada'] = (

    vulnerabilidades['origem']

    .astype(str)

    .str.strip()

    .replace({

        'CloudSecurity': 'Cloud Security',

        'Cloud-Security': 'Cloud Security'

    }))

vulnerabilidades['origem_tratada'].value_counts(dropna=False)

,count
origem_tratada,
Veracode,522
Pentest,507
Cloud Security,506
Qualys,465


In [11]:
ativos_validos = set(ativos['ativo_id'])

vulnerabilidades['ativo_inexistente'] = (

    ~vulnerabilidades['ativo_id'].isin(ativos_validos)

).astype(int)

vulnerabilidades['ativo_inexistente'].value_counts()

,count
ativo_inexistente,
0,1995
1,5


In [12]:
vulnerabilidades['data_abertura_tratada'] = pd.to_datetime(

    vulnerabilidades['data_abertura'],

    errors='coerce',

    dayfirst=True

)

vulnerabilidades['data_abertura_tratada'].head()



,data_abertura_tratada
0,2026-03-21
1,2026-06-01
2,2026-08-18
3,2026-06-16
4,2026-02-12


In [13]:
vulnerabilidades['data_invalida'] = (

    vulnerabilidades['data_abertura_tratada']

    .isna()

    .astype(int)

)

vulnerabilidades['data_invalida'].value_counts()



,count
data_invalida,
0,1922
1,78


In [14]:
df_risco = vulnerabilidades.merge(

    ativos,

    on="ativo_id",

    how="left"

)
df_risco.shape

(2000, 20)

In [15]:
df_risco['severidade_tratada'].value_counts()



,count
severidade_tratada,
Média,506
Crítica,501
Alta,488
Baixa,479
Não Informado,26


In [16]:
pd.crosstab(

    df_risco["criticidade_tratada"],

    df_risco["severidade_tratada"]

)

severidade_tratada,Alta,Baixa,Crítica,Média,Não Informado
criticidade_tratada,,,,,
Alta,97,103,114,96,2
Baixa,141,130,142,150,7
Crítica,107,102,103,120,6
Média,135,131,130,127,11
Não Informado,8,11,10,12,0


In [17]:
top_ativos = (

    df_risco

    .groupby('ativo_id')

    .size()

    .reset_index(name='qtd_vulnerabilidades')

    .sort_values('qtd_vulnerabilidades', ascending=False)

)

top_ativos.head(10)

,ativo_id,qtd_vulnerabilidades
85,A0086,20
119,A0120,19
138,A0139,19
192,A0193,18
56,A0057,18
182,A0183,16
32,A0033,15
20,A0021,15
52,A0053,15
132,A0133,15


In [18]:
top_risco = (

    df_risco[

        df_risco["severidade_tratada"].isin(["Crítica", "Alta"])

    ]

    .groupby(

        ["ativo_id", "sistema", "criticidade_tratada"],

        dropna=False

    )

    .agg(

        qtd_criticas_altas=("vuln_id", "count")

    )

    .reset_index()

    .sort_values("qtd_criticas_altas", ascending=False)

)

top_risco.head(10)

,ativo_id,sistema,criticidade_tratada,qtd_criticas_altas
116,A0120,Sistema_120,Alta,13
85,A0086,Sistema_86,Média,13
52,A0053,Sistema_53,Baixa,11
157,A0161,Sistema_161,Crítica,11
58,A0059,Sistema_59,Alta,10
26,A0027,Sistema_27,Baixa,9
81,A0082,Sistema_82,Média,9
13,A0014,Sistema_14,Crítica,9
56,A0057,Sistema_57,Crítica,9
72,A0073,Sistema_73,Crítica,9


In [19]:
top_risco_atual = (

    df_risco[

        (df_risco["severidade_tratada"].isin(["Crítica", "Alta"])) &

        (df_risco["status"].isin(["Aberta", "Em tratamento"])) &

        (df_risco["ativo_inexistente"] == 0)

    ]

    .groupby(["ativo_id", "sistema", "criticidade_tratada"],dropna=False)

    .agg(qtd_risco_atual=("vuln_id", "count"))

    .reset_index()

    .sort_values("qtd_risco_atual", ascending=False)

)

top_risco_atual.head(10)

,ativo_id,sistema,criticidade_tratada,qtd_risco_atual
78,A0086,Sistema_86,Média,8
103,A0120,Sistema_120,Alta,7
60,A0067,Sistema_67,Baixa,7
53,A0059,Sistema_59,Alta,7
142,A0161,Sistema_161,Crítica,7
19,A0024,Sistema_24,Baixa,6
39,A0045,Sistema_45,Média,6
20,A0025,Sistema_25,Média,6
3,A0004,Sistema_4,Alta,6
163,A0183,Sistema_183,Média,6


In [25]:
peso_criticidade = {

    'Crítica': 4,

    'Alta': 3,

    'Média': 2,

    'Baixa': 1,

    'Não Informado': 0

}

peso_severidade = {

    'Crítica': 4,

    'Alta': 3,

    'Média': 2,

    'Baixa': 1,

    'Não Informado': 0

}

df_risco['peso_criticidade'] = (

    df_risco['criticidade_tratada']

    .map(peso_criticidade)

)

df_risco['peso_severidade'] = (

    df_risco['severidade_tratada']

    .map(peso_severidade)

)

df_risco['risk_score'] = (

    df_risco['peso_criticidade']

    * df_risco['peso_severidade']

)

df_risco[['criticidade_tratada',

          'severidade_tratada',

          'risk_score']].head()

,criticidade_tratada,severidade_tratada,risk_score
0,Crítica,Baixa,4.0
1,Média,Alta,6.0
2,Crítica,Alta,12.0
3,Média,Alta,6.0
4,Média,Alta,6.0


In [26]:
df_risco['risk_score'].value_counts().sort_index()

,count
risk_score,
0.0,67
1.0,130
2.0,281
3.0,244
4.0,371
6.0,231
8.0,250
9.0,97
12.0,221


In [ ]:
 por partes,

In [28]:
ativos.to_csv(

    "ativos_tratado.csv",

    sep=";",

    index=False

)

vulnerabilidades.to_csv(

    "vulnerabilidades_tratado.csv",

    sep=";",

    index=False

)

df_risco.to_csv(

    "dataset_analitico_risco.csv",

    sep=";",

    index=False

)

print("Arquivos gerados")

Arquivos gerados


In [29]:
ativos.to_csv(

    "ativos_tratado.csv",

    sep=";",

    index=False,

    encoding="utf-8-sig"

)

vulnerabilidades.to_csv(

    "vulnerabilidades_tratado.csv",

    sep=";",

    index=False,

    encoding="utf-8-sig"

)

df_risco.to_csv(

    "dataset_analitico_risco.csv",

    sep=";",

    index=False,

    encoding="utf-8-sig"

)